In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# =====================================================================
# Reduced LJ units: m = eps = sigma = k_B = 1
# =====================================================================
EPS, SIGMA, MASS = 1.0, 1.0, 1.0

N = 64               # more particles than 12 so g(r) is meaningful
L0 = 16.0            # initial (dilute) box side  -> rho = 0.25
L_FINAL = 8.5        # final box side             -> rho = 0.89
RC_MAX = 2.5         # cutoff; must stay < L/2 (true for L >= 5.1)
DT = 0.005           # retest this for LJ
T_INIT = 0.1         # low starting temperature (compression will heat it; no thermostat)
MIN_SEP = 1.0        # minimum starting separation (avoids huge forces)

COMPRESS_FACTOR = 0.99   # L_new / L_old at each compression stage
RELAX_STEPS = 400        # steps at each new box size (second half is sampled)
SAMPLE_EVERY = 10
FRAME_EVERY = 20         # store a frame for the animation every n steps

rng = np.random.default_rng(42)


# =====================================================================
# PART 1: LJ PHYSICS AND VALIDATION
# =====================================================================
def min_image(d, L):
    """Minimum-image convention for periodic boundaries."""
    return d - L * np.round(d / L)


def lj_potential_pair(r, rc):
    """Truncated-and-energy-shifted LJ potential for separation r."""
    sr6 = (SIGMA / r) ** 6
    u = 4 * EPS * (sr6 ** 2 - sr6)
    src6 = (SIGMA / rc) ** 6
    u_rc = 4 * EPS * (src6 ** 2 - src6)
    return np.where(r < rc, u - u_rc, 0.0)


def lj_forces(pos, L, rc):
    """
    LJ forces on all particles (periodic box, minimum image).
    F_ij = 24 eps / r^2 * [2 (sigma/r)^12 - (sigma/r)^6] * r_ij,  r_ij = r_i - r_j
    Returns (forces (N,2), total potential energy).
    """
    d = min_image(pos[:, None, :] - pos[None, :, :], L)
    r2 = np.sum(d ** 2, axis=2)
    np.fill_diagonal(r2, np.inf)                 # no self-interaction
    mask = r2 < rc ** 2                          # cutoff

    r2s = np.where(mask, r2, 1.0)                # safe value outside cutoff
    sr6 = (SIGMA ** 2 / r2s) ** 3
    f_over_r2 = np.where(mask, 24 * EPS / r2s * (2 * sr6 ** 2 - sr6), 0.0)
    F = np.sum(f_over_r2[:, :, None] * d, axis=1)

    src6 = (SIGMA / rc) ** 6
    u_rc = 4 * EPS * (src6 ** 2 - src6)
    u = 4 * EPS * (sr6 ** 2 - sr6) - u_rc
    U = 0.5 * np.sum(np.where(mask, u, 0.0))     # each pair counted twice
    return F, U


def validate_lj():
    """Required validation checks before any compression."""
    rc, L = RC_MAX, 20.0
    print("---- PART 1: LJ validation ----")

    # 1. Equal and opposite pair forces
    pos = np.array([[5.0, 5.0], [6.3, 5.4]])
    F, _ = lj_forces(pos, L, rc)
    assert np.allclose(F[0], -F[1]), "pair forces not equal and opposite"
    print("1. Pair forces equal and opposite: OK")

    # 2. Analytical force vs numerical derivative of the potential
    r, h = 1.3, 1e-6
    f_analytic = 24 * EPS / r * (2 * (SIGMA / r) ** 12 - (SIGMA / r) ** 6)
    f_numeric = -(lj_potential_pair(r + h, rc) - lj_potential_pair(r - h, rc)) / (2 * h)
    assert np.isclose(f_analytic, f_numeric, rtol=1e-5)
    pos = np.array([[0.0, 0.0], [r, 0.0]])
    F, _ = lj_forces(pos, L, rc)
    assert np.isclose(F[1, 0], f_analytic)        # force on particle 1 along +x
    print(f"2. Analytical vs numerical force at r={r}: {f_analytic:.6f} vs {f_numeric:.6f}: OK")

    # 3. Potential energy and zero force at r_min = 2^(1/6) sigma
    rmin = 2 ** (1 / 6) * SIGMA
    pos = np.array([[0.0, 0.0], [rmin, 0.0]])
    F, U = lj_forces(pos, L, rc)
    assert np.allclose(F, 0.0, atol=1e-10)
    assert np.isclose(U, lj_potential_pair(rmin, rc))
    print(f"3. At r_min: force = 0, U = {U:.4f} (= -eps minus cutoff shift): OK")

    # 4. Total force on an isolated system is zero (Newton's 3rd law)
    pos = init_random_min_sep(N, L0, MIN_SEP)
    F, _ = lj_forces(pos, L0, rc)
    assert np.allclose(F.sum(axis=0), 0.0, atol=1e-9)
    print("4. Sum of all forces = 0: OK")

    # 5. Starting configuration has no extremely close pairs
    d = min_image(pos[:, None, :] - pos[None, :, :], L0)
    dist = np.linalg.norm(d, axis=2) + np.eye(N) * 1e9
    assert dist.min() >= MIN_SEP
    print(f"5. Smallest starting separation = {dist.min():.3f} >= {MIN_SEP}: OK")


# =====================================================================
# PART 2: COMPRESSION AND SIMULATION
# =====================================================================
def init_random_min_sep(N, L, min_sep):
    """Random positions, rejecting any particle closer than min_sep."""
    pts = []
    while len(pts) < N:
        p = rng.uniform(0, L, size=2)
        if all(np.linalg.norm(min_image(p - q, L)) >= min_sep for q in pts):
            pts.append(p)
    return np.array(pts)


def init_velocities(N, T):
    """Gaussian velocities, centre-of-mass removed, scaled to exactly T."""
    v = rng.normal(0, np.sqrt(T / MASS), size=(N, 2))
    v -= v.mean(axis=0)
    return v * np.sqrt(T / temperature(v))


def kinetic_energy(vel):
    return 0.5 * MASS * np.sum(vel ** 2)


def temperature(vel):
    """T = 2K / f with f = 2N - 2 (centre-of-mass motion removed), k_B = 1."""
    f = 2 * len(vel) - 2
    return 2 * kinetic_energy(vel) / f


def get_rc(L):
    """Cutoff stays below half the box length."""
    return min(RC_MAX, 0.49 * L)


def verlet_step(pos, vel, F, L, rc, dt):
    """Velocity Verlet with periodic wrapping. Returns the unwrapped displacement."""
    vel = vel + 0.5 * dt * F / MASS
    disp = vel * dt
    pos = (pos + disp) % L
    F, U = lj_forces(pos, L, rc)
    vel = vel + 0.5 * dt * F / MASS
    return pos, vel, F, U, disp


def compress_box(pos, L, factor):
    """Imposed compression: x' = s x, L' = s L. This is NOT an LJ force."""
    return pos * factor, L * factor


def energy_conservation_test(dts=(0.01, 0.005, 0.0025), t_total=20.0):
    """Fixed box, no thermostat: total energy should be ~constant."""
    print("---- PART 2: fixed-volume energy test (no thermostat) ----")
    pos0 = init_random_min_sep(N, L0, MIN_SEP)
    vel0 = init_velocities(N, T_INIT)
    results = {}
    for dt in dts:
        pos, vel = pos0.copy(), vel0.copy()
        rc = get_rc(L0)
        F, U = lj_forces(pos, L0, rc)
        E0 = kinetic_energy(vel) + U
        E_hist = []
        for _ in range(int(t_total / dt)):
            pos, vel, F, U, _ = verlet_step(pos, vel, F, L0, rc, dt)
            E_hist.append(kinetic_energy(vel) + U)
        E_hist = np.array(E_hist)
        drift = np.max(np.abs(E_hist - E0)) / abs(E0)
        results[dt] = E_hist
        print(f"dt = {dt:<7} max relative energy deviation = {drift:.2e}")
    return results


def run_compression():
    """
    Gradual compression with relaxation at every stage. No thermostat:
    temperature is only MEASURED, so any heating shows up in the data.
    """
    print("---- PART 2: compression run ----")
    pos = init_random_min_sep(N, L0, MIN_SEP)
    vel = init_velocities(N, T_INIT)
    L = L0
    rc = get_rc(L)
    F, U = lj_forces(pos, L, rc)

    hist = {k: [] for k in ["step", "L", "rho", "K", "U", "E", "T"]}
    frames, frame_L = [], []
    stages = []
    step = 0

    while L > L_FINAL + 1e-9:
        # compress, then relax at the new fixed box size
        pos, L = compress_box(pos, L, COMPRESS_FACTOR)
        L = max(L, L_FINAL)
        rc = get_rc(L)
        F, U = lj_forces(pos, L, rc)

        st = {"L": L, "rho": N / L ** 2, "g": [], "T": [], "U": [], "cl": []}
        accum = np.zeros_like(pos)               # unwrapped displacement since sampling began

        for s in range(RELAX_STEPS):
            pos, vel, F, U, disp = verlet_step(pos, vel, F, L, rc, DT)
            step += 1
            K = kinetic_energy(vel)

            if step % SAMPLE_EVERY == 0:
                hist["step"].append(step)
                hist["L"].append(L)
                hist["rho"].append(N / L ** 2)
                hist["K"].append(K)
                hist["U"].append(U)
                hist["E"].append(K + U)
                hist["T"].append(temperature(vel))
            if step % FRAME_EVERY == 0:
                frames.append(pos.copy())
                frame_L.append(L)

            if s >= RELAX_STEPS // 2:            # second half = measurement window
                accum += disp
                if step % SAMPLE_EVERY == 0:
                    st["g"].append(radial_distribution(pos, L, 40, 0.5 * L_FINAL)[1])
                    st["T"].append(temperature(vel))
                    st["U"].append(U / N)
                    st["cl"].append(largest_cluster(pos, L) / N)

        st["g_mean"] = np.mean(st["g"], axis=0)
        st["T_mean"] = np.mean(st["T"])
        st["U_per_N"] = np.mean(st["U"])
        st["cluster"] = np.mean(st["cl"])
        st["msd"] = np.mean(np.sum(accum ** 2, axis=1))
        st["snapshot"] = pos.copy()
        stages.append(st)

    hist = {k: np.array(v) for k, v in hist.items()}
    print(f"Done: {len(stages)} stages, final density = {N / L ** 2:.3f}, "
          f"final T = {hist['T'][-1]:.3f} (started at {T_INIT})")
    return hist, stages, frames, frame_L


def save_animation(frames, frame_L, filename="lj_compression.gif"):
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.set_xlim(-0.5, L0 + 0.5)
    ax.set_ylim(-0.5, L0 + 0.5)
    ax.set_aspect("equal")
    box = plt.Rectangle((0, 0), L0, L0, fill=False, edgecolor="black", lw=2.5)
    ax.add_patch(box)
    scat = ax.scatter(frames[0][:, 0], frames[0][:, 1], s=60,
                      color="royalblue", edgecolor="black")
    hud = ax.text(0.02, 0.95, "", transform=ax.transAxes, fontsize=9,
                  fontfamily="monospace",
                  bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    ax.set_title("Lennard-Jones compression (periodic box)")

    def update(i):
        L = frame_L[i]
        box.set_width(L)
        box.set_height(L)
        scat.set_offsets(frames[i])
        hud.set_text(f"L = {L:.2f}   density = {N / L ** 2:.3f}")
        return scat, box, hud

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=25, blit=True)
    ani.save(filename, writer="pillow", fps=30)
    plt.close(fig)
    print(f"Saved {filename}")


# =====================================================================
# PART 3: ANALYSIS AND VISUALISATION
# =====================================================================
def radial_distribution(pos, L, nbins, rmax):
    """2D g(r): pair counts in rings, divided by the ideal-gas expectation."""
    n = len(pos)
    d = min_image(pos[:, None, :] - pos[None, :, :], L)
    r = np.linalg.norm(d, axis=2)[np.triu_indices(n, 1)]
    counts, edges = np.histogram(r, bins=nbins, range=(0, rmax))
    centers = 0.5 * (edges[1:] + edges[:-1])
    ring_area = np.pi * (edges[1:] ** 2 - edges[:-1] ** 2)
    rho = n / L ** 2
    g = 2 * counts / (n * rho * ring_area)       # factor 2: each pair counted once
    return centers, g


def largest_cluster(pos, L, threshold=1.3 * SIGMA):
    """Size of the largest connected cluster (particles closer than threshold)."""
    n = len(pos)
    d = min_image(pos[:, None, :] - pos[None, :, :], L)
    adj = (np.linalg.norm(d, axis=2) < threshold) & ~np.eye(n, dtype=bool)
    seen = np.zeros(n, dtype=bool)
    best = 0
    for start in range(n):
        if seen[start]:
            continue
        stack, size = [start], 0
        seen[start] = True
        while stack:
            i = stack.pop()
            size += 1
            for j in np.where(adj[i] & ~seen)[0]:
                seen[j] = True
                stack.append(j)
        best = max(best, size)
    return best


def pick_stages(stages, targets=(0.25, 0.5, 0.7, 0.85, 1.0)):
    rhos = np.array([s["rho"] for s in stages])
    return [stages[int(np.argmin(np.abs(rhos - t)))] for t in targets]


def plot_histories(hist, stages, energy_test):
    fig, ax = plt.subplots(2, 3, figsize=(15, 8))

    # (a) fixed-volume energy test
    for dt, E in energy_test.items():
        ax[0, 0].plot(np.arange(len(E)) * dt, E, label=f"dt={dt}")
    ax[0, 0].set(title="Fixed volume, no thermostat: total energy",
                 xlabel="time", ylabel="E")
    ax[0, 0].legend()

    # (b) K, U, E during compression
    t = hist["step"] * DT
    ax[0, 1].plot(t, hist["K"], label="Kinetic")
    ax[0, 1].plot(t, hist["U"], label="Potential")
    ax[0, 1].plot(t, hist["E"], label="Total")
    ax[0, 1].set(title="Energy during compression", xlabel="time", ylabel="energy")
    ax[0, 1].legend()

    # (c) density and temperature history
    ax[0, 2].plot(t, hist["rho"], color="tab:green", label="density")
    ax[0, 2].set(title="Density and measured temperature", xlabel="time", ylabel="density")
    ax2 = ax[0, 2].twinx()
    ax2.plot(t, hist["T"], color="tab:red", alpha=0.7)
    ax2.set_ylabel("T (red)")

    rho = [s["rho"] for s in stages]
    # (d) potential energy per particle vs density
    ax[1, 0].plot(rho, [s["U_per_N"] for s in stages], "o-")
    ax[1, 0].set(title="Potential energy per particle vs density",
                 xlabel="density", ylabel="U / N")

    # (e) largest cluster fraction
    ax[1, 1].plot(rho, [s["cluster"] for s in stages], "o-", color="tab:purple")
    ax[1, 1].set(title="Largest cluster (fraction of N)", xlabel="density",
                 ylabel="size / N")

    # (f) mobility (MSD over the measurement window)
    ax[1, 2].plot(rho, [s["msd"] for s in stages], "o-", color="tab:orange")
    ax[1, 2].set(title="Mobility: MSD in measurement window", xlabel="density",
                 ylabel="MSD")

    plt.tight_layout()
    plt.savefig("lj_histories.png", dpi=150)
    plt.close(fig)
    print("Saved lj_histories.png")


def plot_gr(stages):
    chosen = pick_stages(stages)
    fig, ax = plt.subplots(figsize=(8, 5))
    centers = radial_distribution(chosen[0]["snapshot"], chosen[0]["L"], 40,
                                  0.5 * L_FINAL)[0]
    for s in chosen:
        ax.plot(centers, s["g_mean"], label=f"rho={s['rho']:.2f}, T={s['T_mean']:.2f}")
    ax.axhline(1.0, color="gray", ls="--", lw=0.8)
    ax.set(title="Radial distribution function g(r)", xlabel="r / sigma", ylabel="g(r)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("lj_gr.png", dpi=150)
    plt.close(fig)
    print("Saved lj_gr.png")


def plot_snapshots(stages):
    chosen = pick_stages(stages)
    fig, axes = plt.subplots(1, len(chosen), figsize=(4 * len(chosen), 4.3))
    for ax, s in zip(axes, chosen):
        ax.scatter(s["snapshot"][:, 0], s["snapshot"][:, 1], s=40,
                   color="royalblue", edgecolor="black")
        ax.set_xlim(0, s["L"])
        ax.set_ylim(0, s["L"])
        ax.set_aspect("equal")
        ax.set_title(f"rho = {s['rho']:.2f}")
    plt.tight_layout()
    plt.savefig("lj_snapshots.png", dpi=150)
    plt.close(fig)
    print("Saved lj_snapshots.png")


# =====================================================================
# MAIN
# =====================================================================
if __name__ == "__main__":
    validate_lj()                                   # Part 1
    energy_test = energy_conservation_test()        # Part 2
    hist, stages, frames, frame_L = run_compression()
    save_animation(frames, frame_L)
    plot_histories(hist, stages, energy_test)       # Part 3
    plot_gr(stages)
    plot_snapshots(stages)